# rikfeed — daily market-data feed

End-to-end run of the **rik** pipeline: install from GitHub, define an
instrument universe, collect daily bars from multiple sources, validate,
and persist to a WAL-mode SQLite store.

```
sources (yfinance, FinanceDataReader)  ->  rikcore  ->  standard records  ->  rikfeed (SQLite)
```

Designed to run after the KRX close. Re-running is safe: the store upserts
on `(symbol, timestamp, source)`, so the same day never duplicates.

## 1. Install

Workspace resolution only works for local clones, so on Colab each package
is installed from its subdirectory. `rikschema` first (it's the contract),
then `rikcore`. `rikfeed` lives in this repo too; install it the same way
if you want its store/scheduler, or just clone the repo.

Replace the tag/branch as needed.

In [ ]:
# Install the rik packages from GitHub.
# Using @main here; pin to a tag (e.g. @rikcore-v0.1.0) for reproducibility.
REPO = 'https://github.com/syuririk/rik.git'
REF = 'main'

import sys, subprocess

def pip_install(pkg_subdir):
    url = f'git+{REPO}@{REF}#subdirectory=packages/{pkg_subdir}'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', url],
        check=True,
    )

for sub in ['rikschema', 'rikcore', 'rikfeed']:
    print(f'installing {sub} ...')
    pip_install(sub)

# Source adapters need yfinance + finance-datareader (rikcore 'sources' extra).
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'yfinance', 'finance-datareader', 'pandas'], check=True)
print('done')

## 2. Define the instrument universe

Each `InstrumentMeta` carries the canonical id plus the per-source aliases
the resolver uses. This small demo universe mixes a KR equity (e.g. 005930.KS) with
global index/ETF proxies, all sourced via yfinance —
the building blocks for a cross-sectional, multi-asset panel.

Extend this list toward your full 27-asset set; the pipeline scales with it.

In [ ]:
from rikschema import InstrumentMeta, AssetClass, Currency, SourceId

UNIVERSE = [
    InstrumentMeta(
        symbol='KRX:005930', name='Samsung Electronics',
        asset_class=AssetClass.EQUITY, currency=Currency.KRW,
        aliases={SourceId.FDR: '005930', SourceId.YFINANCE: '005930.KS'},
        country='KR', exchange='KRX',
    ),
    InstrumentMeta(
        symbol='INDEX:KOSPI', name='KOSPI Composite',
        asset_class=AssetClass.EQUITY_INDEX, currency=Currency.NONE,
        aliases={SourceId.YFINANCE: '^KS11'}, country='KR',
    ),
    InstrumentMeta(
        symbol='INDEX:SPX', name='S&P 500',
        asset_class=AssetClass.EQUITY_INDEX, currency=Currency.NONE,
        aliases={SourceId.YFINANCE: '^GSPC'}, country='US',
    ),
    InstrumentMeta(
        symbol='ETF:TLT', name='iShares 20+ Year Treasury Bond ETF',
        asset_class=AssetClass.ETF, currency=Currency.USD,
        aliases={SourceId.YFINANCE: 'TLT'}, country='US', exchange='NASDAQ',
    ),
    InstrumentMeta(
        symbol='ETF:GLD', name='SPDR Gold Shares',
        asset_class=AssetClass.ETF, currency=Currency.USD,
        aliases={SourceId.YFINANCE: 'GLD'}, country='US', exchange='NYSE',
    ),
]

print(f'{len(UNIVERSE)} instruments defined')
for m in UNIVERSE:
    print(f'  {m.symbol:14} {m.name}')

## 3. Wire up the resolver and sources

The resolver indexes the universe. Each source is registered with it, so
the orchestrator can ask every source only for the symbols it can serve
(`filter_supported`) and merge the results under canonical ids.

In [ ]:
from rikcore import SymbolResolver, SourceRegistry, YFinanceSource, FDRSource

resolver = SymbolResolver(UNIVERSE)

registry = SourceRegistry()
registry.register(YFinanceSource(resolver))
registry.register(FDRSource(resolver))   # KR equities via FinanceDataReader

# The orchestrator routes each symbol only to sources that can serve it
# (filter_supported), so KR tickers go to FDR and indices/ETFs to yfinance.
all_symbols = resolver.known_symbols()
for src in registry.all():
    served = resolver.filter_supported(all_symbols, src.source_id)
    print(f'{src.source_id.value:10} -> {served}')

## 4. Collect → validate → store

`StoreEmitter` adapts the SQLite store to the emitter protocol, so the
orchestrator writes straight into the feed. `run()` returns the records,
a validation report, and the emitter's output (here: rows written).

Set the date window. For a daily post-close feed you'd typically pull the
last few days to cover gaps; here we take a trailing window.

In [ ]:
from datetime import date, timedelta
from rikcore import Orchestrator
from rikfeed import SQLiteStore
from rikfeed.storage import StoreEmitter

DB_PATH = 'rikfeed.db'
end = date.today()
start = end - timedelta(days=10)   # trailing window; upsert handles overlap

store = SQLiteStore(DB_PATH)
orch = Orchestrator(registry, StoreEmitter(store))

result = orch.run(all_symbols, start, end)

print('validation:', result.report.summary())
print('records collected:', len(result.records))
print('rows written to store:', result.output)
print('total rows in store:', store.count())

## 5. Inspect the stored data

Read straight from SQLite into a DataFrame to sanity-check what landed.

In [ ]:
import pandas as pd
import sqlite3

con = sqlite3.connect(DB_PATH)
df = pd.read_sql_query(
    'SELECT symbol, timestamp, source, close, currency '
    'FROM ohlcv ORDER BY symbol, timestamp', con,
)
con.close()

print('rows:', len(df))
print('symbols:', df['symbol'].nunique())
df.groupby('symbol').tail(1)   # latest bar per instrument

## 6. Build a close-price panel

Pivot to a wide `date × symbol` close-price matrix — the shape a
cross-sectional regression across assets consumes directly.

In [ ]:
panel = df.pivot_table(index='timestamp', columns='symbol', values='close')
panel.index = pd.to_datetime(panel.index)
panel = panel.sort_index()
print('panel shape (dates x assets):', panel.shape)
panel.tail()

## 7. Scheduling (optional)

`rikfeed.next_run_after` computes the next KRX-close-plus-margin slot,
skipping weekends. Colab can't host a long-lived scheduler, but you can use
this to decide whether a run is due, or wire it into a cron/GitHub Action.

In [ ]:
from rikfeed import next_run_after
from datetime import datetime
from zoneinfo import ZoneInfo

now = datetime.now(ZoneInfo('Asia/Seoul'))
print('now (KST):     ', now.strftime('%Y-%m-%d %H:%M %A'))
print('next feed run: ', next_run_after(now).strftime('%Y-%m-%d %H:%M %A'))

---

**Re-run safety.** Running this notebook again upserts the same keys rather
than duplicating. **Next steps:** grow `UNIVERSE` toward the full 27-asset
set, add an OECD/macro source for `MacroRecord` series, or point the store
at a path on Google Drive to persist across Colab sessions.